<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_alt_medium_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — alternating medium (Shakespeare + TinyStories char)

Companion to `train_dual_medium_sts.ipynb`. Same model size and iter budget, but uses the **original alternating** training scheme (pure-per-corpus passes with masked-gradient slot routing) and Beta(0.5, 0.5) alpha sampling, instead of alt_mixed Uniform. On the wiki experiments these two recipes tied within noise at the compact scale, so we want to rerun the head-to-head on tinystories.

- `n_layer=6, n_embd=512` (~25M total params, ~12.5M per slot)
- `max_iters=12000`
- `dropout=0.2` (alternating doesn't get alpha-stochasticity regularization, so it needs conventional dropout)
- `learning_rate=8e-4`, `gradient_accumulation_steps=2` (effective batch 128)
- `batch_mode=alternating`, `mix_distribution=beta_half`

Compute estimate: ~2 hours on T4.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Run directory (tagged `-alt-medium-sts` so this run is distinct from the others)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-alt-medium-sts"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-alt-medium-sts'
elif not RUN_ID.endswith('-alt-medium-sts'):
    RUN_ID = RUN_ID + '-alt-medium-sts'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

On a T4 GPU expect roughly 2 hours for 12000 iters. Pass time will be ~55-60s/pass.

In [ ]:
!python train_dual.py config/train_shakespeare_tinystories_dual_alt_medium.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alternating \
    --first_pass_corpus=shake \
    --mix_distribution=beta_half